# Bangla Deepfake Inference Comparison

Runs inference on `Data/` with `best_lstm_model.pth`, `best_wave2vec_model.pt`, and `best_xlsr_model.pt`, then compares outputs graphically.

In [ ]:
import math
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import WavLMModel, Wav2Vec2Model

sns.set_theme(style='whitegrid')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

ROOT = Path.cwd()
DATA_DIR = ROOT / 'Data'
LSTM_MODEL_PATH = ROOT / 'best_lstm_model.pth'
WAVE2VEC_MODEL_PATH = ROOT / 'best_wave2vec_model.pt'
XLSR_MODEL_PATH = ROOT / 'best_xlsr_model.pt'

for p in [DATA_DIR, LSTM_MODEL_PATH, WAVE2VEC_MODEL_PATH, XLSR_MODEL_PATH]:
    print(f'{p} ->', 'OK' if p.exists() else 'MISSING')

SAMPLE_RATE = 16000
AUDIO_DURATION = 5
MAX_AUDIO_LEN = SAMPLE_RATE * AUDIO_DURATION
N_MFCC = 40
MAX_LEN_MFCC = 157
NUM_CLASSES = 2
AUDIO_EXTS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac'}

def safe_torch_load(path, device, weights_only=False):
    try:
        return torch.load(str(path), map_location=device, weights_only=weights_only)
    except TypeError:
        return torch.load(str(path), map_location=device)

def load_state_from_checkpoint(model, checkpoint):
    if isinstance(checkpoint, dict):
        for key in ('model_state_dict', 'state_dict'):
            if key in checkpoint and isinstance(checkpoint[key], dict):
                model.load_state_dict(checkpoint[key])
                return key
    model.load_state_dict(checkpoint)
    return None

def iter_audio_files(root):
    return sorted([p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in AUDIO_EXTS])

def load_audio(filepath, sr=SAMPLE_RATE, duration=AUDIO_DURATION):
    y, _ = librosa.load(str(filepath), sr=sr, duration=duration)
    target = sr * duration
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)), mode='constant')
    else:
        y = y[:target]
    return y

def extract_mfcc(filepath):
    y = load_audio(filepath)
    mfcc = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC)
    if mfcc.shape[1] < MAX_LEN_MFCC:
        mfcc = np.pad(mfcc, ((0, 0), (0, MAX_LEN_MFCC - mfcc.shape[1])), mode='constant')
    else:
        mfcc = mfcc[:, :MAX_LEN_MFCC]
    return mfcc.T.astype(np.float32)

def extract_waveform(filepath):
    y = load_audio(filepath).astype(np.float32)
    y = (y - y.mean()) / (y.std() + 1e-7)
    return y

class LSTMModel(nn.Module):
    def __init__(self, input_size=N_MFCC, hidden_size=128, num_layers=2, num_classes=2, dropout=0.5):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.dropout(out)
        out = F.relu(self.fc1(out))
        out = self.dropout(out)
        return self.fc2(out)

WAVLM_MODEL_NAME = 'microsoft/wavlm-large'
WAVLM_DIM = 1024
WAVLM_FREEZE_LAYERS = 12

XLSR_MODEL_NAME = 'facebook/wav2vec2-xls-r-300m'
XLSR_DIM = 1024
XLSR_FREEZE_LAYERS = 12

AASIST_HIDDEN = 160
AASIST_EMB_DIM = 128
AASIST_NUM_HEADS = 4
AASIST_NUM_GAT_LAYERS = 2
NUM_SPECTRAL_NODES = 32
DROPOUT = 0.3
OC_MARGIN = 0.5
OC_SCALE = 10.0

class ModifiedAASIST(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=160, emb_dim=128, num_heads=4, num_gat_layers=2, num_spectral_nodes=32, max_seq_len=512, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim))
        self.pos_enc = nn.Parameter(torch.randn(1, max_seq_len, hidden_dim) * 0.02)
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, 3, padding=1, groups=hidden_dim),
            nn.Conv1d(hidden_dim, hidden_dim, 1),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
        )
        self.spectral_basis = nn.Parameter(torch.randn(num_spectral_nodes, hidden_dim) * 0.02)
        self.spectral_proj = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim))
        self.drop = nn.Dropout(dropout)
        self.gat_layers = nn.ModuleList()
        self.gat_norms1 = nn.ModuleList()
        self.gat_ffn = nn.ModuleList()
        self.gat_norms2 = nn.ModuleList()
        for _ in range(num_gat_layers):
            self.gat_layers.append(nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True))
            self.gat_norms1.append(nn.LayerNorm(hidden_dim))
            self.gat_ffn.append(nn.Sequential(nn.Linear(hidden_dim, hidden_dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim * 4, hidden_dim), nn.Dropout(dropout)))
            self.gat_norms2.append(nn.LayerNorm(hidden_dim))
        self.readout_query = nn.Parameter(torch.randn(1, 1, hidden_dim) * 0.02)
        self.readout_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.readout_norm = nn.LayerNorm(hidden_dim)
        self.output_fc = nn.Sequential(nn.Linear(hidden_dim, emb_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(emb_dim, emb_dim))

    def forward(self, x):
        bsz, tlen, _ = x.shape
        h = self.input_proj(x) + self.pos_enc[:, :tlen, :]
        h = self.drop(h)
        h = h + self.temporal_conv(h.transpose(1, 2)).transpose(1, 2)
        s_nodes = self.spectral_basis.unsqueeze(0).expand(bsz, -1, -1)
        w = F.softmax(torch.bmm(s_nodes, h.transpose(1, 2)) / math.sqrt(h.size(-1)), dim=-1)
        spectral = self.spectral_proj(torch.bmm(w, h))
        nodes = torch.cat([h, spectral], dim=1)
        for gat, n1, ffn, n2 in zip(self.gat_layers, self.gat_norms1, self.gat_ffn, self.gat_norms2):
            res = nodes
            out, _ = gat(n1(nodes), n1(nodes), n1(nodes))
            nodes = res + self.drop(out)
            nodes = nodes + ffn(n2(nodes))
        q = self.readout_query.expand(bsz, -1, -1)
        out, attn = self.readout_attn(q, nodes, nodes)
        emb = self.readout_norm(out).squeeze(1)
        return self.output_fc(emb), attn

class OCSoftmaxLoss(nn.Module):
    def __init__(self, feat_dim=128, margin=0.5, scale=10.0):
        super().__init__()
        self.w = nn.Parameter(torch.empty(feat_dim))
        nn.init.normal_(self.w, mean=0, std=0.01)
        self.margin = margin
        self.scale = scale

    def score(self, x):
        return F.cosine_similarity(F.normalize(x, dim=1), F.normalize(self.w.unsqueeze(0), dim=1).expand_as(x), dim=1)

    def forward(self, x, labels):
        cos_theta = self.score(x)
        is_real = (labels == 0).float()
        is_fake = (labels == 1).float()
        logits = is_real * self.scale * (cos_theta - self.margin) + is_fake * self.scale * (self.margin - cos_theta)
        loss = F.binary_cross_entropy_with_logits(logits, torch.ones_like(logits))
        return loss, cos_theta

class BanglaDeepfakeDetectorWavLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.wavlm = WavLMModel.from_pretrained(WAVLM_MODEL_NAME)
        self._freeze_wavlm()
        self.aasist = ModifiedAASIST(input_dim=WAVLM_DIM, hidden_dim=AASIST_HIDDEN, emb_dim=AASIST_EMB_DIM, num_heads=AASIST_NUM_HEADS, num_gat_layers=AASIST_NUM_GAT_LAYERS, num_spectral_nodes=NUM_SPECTRAL_NODES, dropout=DROPOUT)
        self.oc_softmax = OCSoftmaxLoss(feat_dim=AASIST_EMB_DIM, margin=OC_MARGIN, scale=OC_SCALE)

    def _freeze_wavlm(self):
        self.wavlm.feature_extractor._freeze_parameters()
        for p in self.wavlm.feature_projection.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.wavlm.encoder.layers):
            if i < WAVLM_FREEZE_LAYERS:
                for p in layer.parameters():
                    p.requires_grad = False

    def forward(self, audio, labels=None):
        feats = self.wavlm(input_values=audio).last_hidden_state
        emb, attn = self.aasist(feats)
        if labels is not None:
            loss, scores = self.oc_softmax(emb, labels)
            return loss, scores, emb, attn
        return self.oc_softmax.score(emb), emb, attn

class BanglaDeepfakeDetectorXLSR(nn.Module):
    def __init__(self):
        super().__init__()
        self.xlsr = Wav2Vec2Model.from_pretrained(XLSR_MODEL_NAME)
        self._freeze_xlsr()
        self.aasist = ModifiedAASIST(input_dim=XLSR_DIM, hidden_dim=AASIST_HIDDEN, emb_dim=AASIST_EMB_DIM, num_heads=AASIST_NUM_HEADS, num_gat_layers=AASIST_NUM_GAT_LAYERS, num_spectral_nodes=NUM_SPECTRAL_NODES, dropout=DROPOUT)
        self.oc_softmax = OCSoftmaxLoss(feat_dim=AASIST_EMB_DIM, margin=OC_MARGIN, scale=OC_SCALE)

    def _freeze_xlsr(self):
        self.xlsr.feature_extractor._freeze_parameters()
        for p in self.xlsr.feature_projection.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.xlsr.encoder.layers):
            if i < XLSR_FREEZE_LAYERS:
                for p in layer.parameters():
                    p.requires_grad = False

    def forward(self, audio, labels=None):
        feats = self.xlsr(input_values=audio).last_hidden_state
        emb, attn = self.aasist(feats)
        if labels is not None:
            loss, scores = self.oc_softmax(emb, labels)
            return loss, scores, emb, attn
        return self.oc_softmax.score(emb), emb, attn

lstm_model = LSTMModel().to(DEVICE)
lstm_state = safe_torch_load(LSTM_MODEL_PATH, DEVICE, weights_only=True)
if isinstance(lstm_state, dict) and 'state_dict' in lstm_state:
    lstm_state = lstm_state['state_dict']
lstm_model.load_state_dict(lstm_state)
lstm_model.eval()
print('Loaded LSTM model.')

wave2vec_model = BanglaDeepfakeDetectorWavLM().to(DEVICE)
wave2vec_ckpt = safe_torch_load(WAVE2VEC_MODEL_PATH, DEVICE, weights_only=False)
wave2vec_key = load_state_from_checkpoint(wave2vec_model, wave2vec_ckpt)
wave2vec_model.eval()
print('Loaded Wave2Vec/WavLM model.', f'(key={wave2vec_key})' if wave2vec_key else '')

xlsr_model = BanglaDeepfakeDetectorXLSR().to(DEVICE)
xlsr_ckpt = safe_torch_load(XLSR_MODEL_PATH, DEVICE, weights_only=False)
xlsr_key = load_state_from_checkpoint(xlsr_model, xlsr_ckpt)
xlsr_model.eval()
print('Loaded XLS-R model.', f'(key={xlsr_key})' if xlsr_key else '')

@torch.no_grad()
def predict_lstm(path):
    mfcc = extract_mfcc(path)
    x = torch.from_numpy(mfcc).unsqueeze(0).to(DEVICE)
    logits = lstm_model(x)
    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    pred = int(np.argmax(probs))
    return pred, float(probs[1]), float(probs[0])

@torch.no_grad()
def predict_wave2vec(path):
    audio = extract_waveform(path)
    x = torch.from_numpy(audio).unsqueeze(0).to(DEVICE)
    score, _, _ = wave2vec_model(x)
    s = float(score[0].cpu())
    p_real = float(torch.sigmoid(torch.tensor(s)).item())
    p_fake = 1.0 - p_real
    pred = 0 if s > 0 else 1
    return pred, p_fake, p_real, s

@torch.no_grad()
def predict_xlsr(path):
    audio = extract_waveform(path)
    x = torch.from_numpy(audio).unsqueeze(0).to(DEVICE)
    score, _, _ = xlsr_model(x)
    s = float(score[0].cpu())
    p_real = float(torch.sigmoid(torch.tensor(s)).item())
    p_fake = 1.0 - p_real
    pred = 0 if s > 0 else 1
    return pred, p_fake, p_real, s

label_from_folder = {
    'real_samples': 0,
    'baglafake_deepfake': 1,
    'crikk_deepfake': 1,
    'gemini_deepfake': 1,
}

rows = []
files = iter_audio_files(DATA_DIR)
print('Audio files found:', len(files))

for p in files:
    folder = p.parent.name
    y_true = label_from_folder.get(folder, np.nan)
    try:
        lstm_pred, lstm_p_fake, lstm_p_real = predict_lstm(p)
        wav_pred, wav_p_fake, wav_p_real, wav_score = predict_wave2vec(p)
        xlsr_pred, xlsr_p_fake, xlsr_p_real, xlsr_score = predict_xlsr(p)
        rows.append({
            'filepath': str(p),
            'folder': folder,
            'true_label': y_true,
            'lstm_pred': lstm_pred,
            'lstm_prob_fake': lstm_p_fake,
            'lstm_prob_real': lstm_p_real,
            'wave2vec_pred': wav_pred,
            'wave2vec_prob_fake': wav_p_fake,
            'wave2vec_prob_real': wav_p_real,
            'wave2vec_score': wav_score,
            'xlsr_pred': xlsr_pred,
            'xlsr_prob_fake': xlsr_p_fake,
            'xlsr_prob_real': xlsr_p_real,
            'xlsr_score': xlsr_score,
            'agree_lstm_wave2vec': int(lstm_pred == wav_pred),
            'agree_lstm_xlsr': int(lstm_pred == xlsr_pred),
            'agree_wave2vec_xlsr': int(wav_pred == xlsr_pred),
            'models_agree_all': int(lstm_pred == wav_pred == xlsr_pred),
        })
    except Exception as exc:
        rows.append({'filepath': str(p), 'folder': folder, 'true_label': y_true, 'error': repr(exc)})

df = pd.DataFrame(rows)
display(df.head())
out_csv = ROOT / 'model_comparison_predictions.csv'
df.to_csv(out_csv, index=False)
print('Saved:', out_csv)


c:\Users\Asus\miniconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Device: cuda
f:\ML Project\Bangla Audio Deepfake Detection\Inference\Checkpoint 4\Data -> OK
f:\ML Project\Bangla Audio Deepfake Detection\Inference\Checkpoint 4\best_lstm_model.pth -> OK
f:\ML Project\Bangla Audio Deepfake Detection\Inference\Checkpoint 4\best_wave2vec_model.pt -> OK
f:\ML Project\Bangla Audio Deepfake Detection\Inference\Checkpoint 4\best_xlsr_model.pt -> OK
Loaded LSTM model.
Loaded Wave2Vec/WavLM model. (key=model_state_dict)
Loaded XLS-R model. (key=model_state_dict)
Audio files found: 5311


In [ ]:
from pathlib import Path

df_ok = df[df.get('error').isna()] if 'error' in df.columns else df.copy()
print('Usable rows:', len(df_ok), '/', len(df))

if len(df_ok) == 0:
    raise RuntimeError('No successful predictions. Check errors in df.')

# Output folder for plots
PLOT_DIR = Path.cwd() / "output"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("Saving plots to:", PLOT_DIR)

# 1) Mean fake probability by folder
plot_df = df_ok.melt(
    id_vars=['folder'],
    value_vars=['lstm_prob_fake', 'wave2vec_prob_fake', 'xlsr_prob_fake'],
    var_name='model',
    value_name='fake_probability'
)
plot_df['model'] = plot_df['model'].map({
    'lstm_prob_fake': 'LSTM',
    'wave2vec_prob_fake': 'Wave2Vec/WavLM',
    'xlsr_prob_fake': 'XLS-R + AASIST'
})

fig1, ax1 = plt.subplots(figsize=(11, 5.5))
sns.barplot(data=plot_df, x='folder', y='fake_probability', hue='model', errorbar='sd', ax=ax1)
ax1.set_ylim(0, 1)
ax1.set_title('Mean Fake Probability by Folder')
ax1.set_xlabel('Folder')
ax1.set_ylabel('Fake Probability')
ax1.tick_params(axis='x', rotation=15)
fig1.tight_layout()
fig1.savefig(PLOT_DIR / "mean_fake_probability_by_folder.png", dpi=300, bbox_inches='tight')
plt.show()

# 2) Prediction counts by model
pred_df = pd.concat([
    pd.DataFrame({'model': 'LSTM', 'pred': df_ok['lstm_pred']}),
    pd.DataFrame({'model': 'Wave2Vec/WavLM', 'pred': df_ok['wave2vec_pred']}),
    pd.DataFrame({'model': 'XLS-R + AASIST', 'pred': df_ok['xlsr_pred']})
] , ignore_index=True)
pred_df['pred_label'] = pred_df['pred'].map({0: 'Real', 1: 'Fake'})

fig2, ax2 = plt.subplots(figsize=(9.5, 4.8))
sns.countplot(data=pred_df, x='model', hue='pred_label', ax=ax2)
ax2.set_title('Predicted Class Distribution')
ax2.set_xlabel('Model')
ax2.set_ylabel('Count')
fig2.tight_layout()
fig2.savefig(PLOT_DIR / "predicted_class_distribution.png", dpi=300, bbox_inches='tight')
plt.show()

# 3) Confusion matrices (if ground truth is available from folder names)
if df_ok['true_label'].notna().all():
    y_true = df_ok['true_label'].astype(int).to_numpy()
    y_lstm = df_ok['lstm_pred'].astype(int).to_numpy()
    y_wav = df_ok['wave2vec_pred'].astype(int).to_numpy()
    y_xlsr = df_ok['xlsr_pred'].astype(int).to_numpy()

    def confusion_np(y_t, y_p):
        cm = np.zeros((2, 2), dtype=int)
        for t, p in zip(y_t, y_p):
            cm[t, p] += 1
        return cm

    cm_lstm = confusion_np(y_true, y_lstm)
    cm_wav = confusion_np(y_true, y_wav)
    cm_xlsr = confusion_np(y_true, y_xlsr)

    fig3, axes = plt.subplots(1, 3, figsize=(16.5, 4.7))
    sns.heatmap(
        cm_lstm, annot=True, fmt='d', cmap='Blues', cbar=False,
        xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'], ax=axes[0]
    )
    axes[0].set_title('LSTM Confusion Matrix')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')

    sns.heatmap(
        cm_wav, annot=True, fmt='d', cmap='Greens', cbar=False,
        xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'], ax=axes[1]
    )
    axes[1].set_title('Wave2Vec/WavLM Confusion Matrix')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')

    sns.heatmap(
        cm_xlsr, annot=True, fmt='d', cmap='Oranges', cbar=False,
        xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'], ax=axes[2]
    )
    axes[2].set_title('XLS-R + AASIST Confusion Matrix')
    axes[2].set_xlabel('Predicted')
    axes[2].set_ylabel('True')

    fig3.tight_layout()
    fig3.savefig(PLOT_DIR / "confusion_matrices_lstm_wave2vec_xlsr.png", dpi=300, bbox_inches='tight')
    plt.show()

    print('LSTM accuracy           :', float((y_lstm == y_true).mean()))
    print('Wave2Vec/WavLM accuracy:', float((y_wav == y_true).mean()))
    print('XLS-R + AASIST accuracy:', float((y_xlsr == y_true).mean()))
    if 'agree_lstm_wave2vec' in df_ok.columns:
        print('LSTM vs Wave2Vec agree :', float(df_ok['agree_lstm_wave2vec'].mean()))
    if 'agree_lstm_xlsr' in df_ok.columns:
        print('LSTM vs XLS-R agree    :', float(df_ok['agree_lstm_xlsr'].mean()))
    if 'agree_wave2vec_xlsr' in df_ok.columns:
        print('Wave2Vec vs XLS-R agree:', float(df_ok['agree_wave2vec_xlsr'].mean()))
    if 'models_agree_all' in df_ok.columns:
        print('All-3 models agree     :', float(df_ok['models_agree_all'].mean()))
else:
    print('true_label is missing for some folders; skipped confusion matrices.')